# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rimlazrek1/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

## Setup (Local)

In [1]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
# Same windows as Week 5: Mar–Apr features → May label.
FACT_PREV = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    "])"
)
FACT_NEXT = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-05/*.parquet')"

print("Connected.")
print("  FACT_PREV = month=2026-03 + 2026-04 (features)")
print("  FACT_NEXT = month=2026-05 (label)")


Connected.
  FACT_PREV = month=2026-03 + 2026-04 (features)
  FACT_NEXT = month=2026-05 (label)


## 1. Two paper findings + my methodology questions

Source: `docs/flyrank-seo-research-march-2026.pdf` (*The State of AI-Driven SEO*, March 2026). Constructive audit only — how to make the claims stronger, not a gotcha.

### Finding #1 — Anatomy of Growing Content 

**What they claim:** Rising-impression pages look different from falling ones: longer (≈3.2K vs 2.3K words), younger (≈184 vs 230 days), slightly better position. Large n (≈74K up vs ≈45K down).

**Where does the label/group come from?**  
`up` / `down` from **trend direction**: 30-day impressions vs previous 30 days (>10% up / >10% down). That is a **rule on recent impression change**, not an experiment and not “this page will keep growing.”

**Does the validation design carry the claim?**  
**Partly.** A big observational table supports “growing and declining cohorts **differ** on length/age.” It does **not** by itself support “make pages longer → they will grow.” The paper already tags this as observational / directional — good. To strengthen it: hold client/site constant (grouped comparison), or track the **same** pages before/after an expand-content action (time-aware outcome).

### Finding #3 — Click Capture by Position Tier 

**What they claim:** Weighted CTR falls by position tier (Top 3 ≈0.42% → Deep ≈0.05%). Page-one refinement is a better click bet than spreading effort on deep pages.

**Where does the label/group come from?**  
Groups are **`position_tier` from average position**. Metric is **portfolio weighted CTR** (total clicks ÷ total impressions in the tier) — not a model score, not a per-row average.

**Does the validation design carry the claim?**  
**Yes for the measured pattern** (“CTR is associated with tier in this portfolio”). **Not for causal advice** (“refine snippets → clicks will rise”) unless they later A/B or track revised vs control pages. This finding is the closest cousin of my Lane 4 work: compare CTR **within** tier, don’t treat raw CTR alone as the signal.


## 2. My model under an honest split (before/after)

**Same Week-5 model** on Mar–Apr → May (`imp_prev >= 500`): logistic ranked by `P(is_ctr_underperformer)`. Precision@K next to base rate.

**BEFORE → AFTER:** random page split → client `GroupKFold` (4 folds). Same-site pages share niche/CMS/tracking, so random splits can inflate skill; the honest question is clients the model never saw. Time split is already in the frame (past features, May label).


In [3]:
LABEL = "is_ctr_underperformer"
FEATURE_COLS = ["imp_prev", "ctr_prev", "pos_avg_prev", "engagement_rate_prev", "position_tier"]
NUM_COLS = ["imp_prev", "ctr_prev", "pos_avg_prev", "engagement_rate_prev"]
RANDOM_STATE = 42
IMP_FLOOR = 500
N_FOLDS = 4
KS = (10, 20, 50)


def precision_at_k(scores, labels, k, tie_break=None):
    """Rank by score desc, then tie_break desc (same contract for rule and model)."""
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels)
    if tie_break is None:
        order = np.argsort(-scores, kind="mergesort")
    else:
        tb = np.asarray(tie_break, dtype=float)
        order = np.lexsort((-tb, -scores))
    k = min(k, len(order))
    return float(labels[order[:k]].mean())


def make_pipeline():
    preprocessor = ColumnTransformer(
        [
            ("num", StandardScaler(), NUM_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["position_tier"]),
        ]
    )
    return Pipeline(
        [
            ("prep", preprocessor),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


def eval_on_eligible(train_df, test_df, split_name):
    """Fit on train; score Precision@K on eligible test pages only."""
    model = make_pipeline()
    model.fit(train_df[FEATURE_COLS], train_df[LABEL])
    test_df = test_df.copy()
    test_df["model_score"] = model.predict_proba(test_df[FEATURE_COLS])[:, 1]
    eligible = test_df[test_df["imp_prev"] >= IMP_FLOOR].copy()
    y = eligible[LABEL].to_numpy()
    imp = eligible["imp_prev"].to_numpy()
    row = {
        "split": split_name,
        "n_train": int(len(train_df)),
        "n_test": int(len(eligible)),
        "n_test_clients": int(eligible["client_hash_id"].nunique()),
        "base_rate": float(y.mean()) if len(eligible) else float("nan"),
    }
    for k in KS:
        row[f"precision_at_{k}"] = precision_at_k(
            eligible["model_score"].to_numpy(), y, k, tie_break=imp
        )
    return row, model, eligible


# --- Same Mar–Apr → May frame as Week 5 ---
features = con.sql(f"""
    WITH prev_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_prev,
            SUM(gsc_clicks) AS clk_prev,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_prev,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sessions_prev,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS engaged_prev
        FROM {FACT_PREV}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    prev_scored AS (
        SELECT
            *,
            CASE WHEN imp_prev > 0 THEN 100.0 * clk_prev / imp_prev END AS ctr_prev,
            CASE
                WHEN sessions_prev > 0 THEN 100.0 * engaged_prev / sessions_prev
            END AS engagement_rate_prev,
            CASE
                WHEN pos_avg_prev <= 3 THEN 'top_3'
                WHEN pos_avg_prev <= 10 THEN 'page_1'
                WHEN pos_avg_prev <= 20 THEN 'striking'
                WHEN pos_avg_prev <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier
        FROM prev_daily
        WHERE pos_avg_prev > 0
    ),
    prev_ranked AS (
        SELECT
            *,
            MEDIAN(ctr_prev) OVER (PARTITION BY position_tier) AS tier_median_ctr
        FROM prev_scored
    ),
    next_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_next,
            SUM(gsc_clicks) AS clk_next,
            AVG(NULLIF(gsc_avg_position, 0)) AS pos_avg_next
        FROM {FACT_NEXT}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 500
    ),
    next_scored AS (
        SELECT
            *,
            CASE WHEN imp_next > 0 THEN 100.0 * clk_next / imp_next END AS ctr_next,
            CASE
                WHEN pos_avg_next <= 3 THEN 'top_3'
                WHEN pos_avg_next <= 10 THEN 'page_1'
                WHEN pos_avg_next <= 20 THEN 'striking'
                WHEN pos_avg_next <= 50 THEN 'page_3_5'
                ELSE 'deep'
            END AS position_tier_next
        FROM next_daily
        WHERE pos_avg_next > 0
    ),
    next_labeled AS (
        SELECT
            *,
            MEDIAN(ctr_next) OVER (PARTITION BY position_tier_next) AS tier_median_ctr_next,
            CASE
                WHEN ctr_next < MEDIAN(ctr_next) OVER (PARTITION BY position_tier_next)
                     AND imp_next >= 500
                THEN 1
                ELSE 0
            END AS is_ctr_underperformer
        FROM next_scored
    )
    SELECT
        p.*,
        n.imp_next,
        n.ctr_next,
        n.position_tier_next,
        n.tier_median_ctr_next,
        n.is_ctr_underperformer
    FROM prev_ranked p
    INNER JOIN next_labeled n
        USING (client_hash_id, content_hash_id)
""").df()

model_df = features.dropna(subset=["ctr_prev", "pos_avg_prev"]).copy()
model_df["engagement_rate_prev"] = model_df["engagement_rate_prev"].fillna(0)
model_df["baseline_score"] = model_df["tier_median_ctr"] - model_df["ctr_prev"]
# Past ctr_gap = baseline score (fair comparator only). May ctr_gap_next encodes the label — trap only.
model_df["ctr_gap"] = model_df["baseline_score"]
model_df["ctr_gap_next"] = model_df["tier_median_ctr_next"] - model_df["ctr_next"]

print(f"Frame n = {len(model_df):,} | clients = {model_df['client_hash_id'].nunique():,}")
print(f"Slice: features=2026-03+2026-04; label=2026-05 | base rate = {model_df[LABEL].mean():.3f}")

# --- BEFORE: random page split ---
train_rand, test_rand = train_test_split(
    model_df, test_size=0.25, random_state=RANDOM_STATE, stratify=model_df[LABEL]
)
row_before, _, _ = eval_on_eligible(train_rand, test_rand, "BEFORE: random page split")

# --- AFTER: client GroupKFold (honest — same as Week 5) ---
gkf = GroupKFold(n_splits=N_FOLDS)
groups = model_df["client_hash_id"].to_numpy()
fold_rows = []
oos_parts = []

for fold, (tr_idx, te_idx) in enumerate(
    gkf.split(model_df[FEATURE_COLS], model_df[LABEL], groups), start=1
):
    train_df = model_df.iloc[tr_idx]
    test_df = model_df.iloc[te_idx]
    row, _, eligible = eval_on_eligible(
        train_df, test_df, f"AFTER: GroupKFold fold {fold}"
    )
    row["fold"] = fold
    fold_rows.append(row)
    oos_parts.append(eligible.assign(fold=fold))

fold_df = pd.DataFrame(fold_rows)
oos_eligible = pd.concat(oos_parts, ignore_index=True)

row_after = {
    "split": "AFTER: client GroupKFold (mean)",
    "n_train": int(fold_df["n_train"].mean()),
    "n_test": int(fold_df["n_test"].mean()),
    "n_test_clients": float(fold_df["n_test_clients"].mean()),
    "base_rate": float(fold_df["base_rate"].mean()),
}
for k in KS:
    row_after[f"precision_at_{k}"] = float(fold_df[f"precision_at_{k}"].mean())

comparison = pd.DataFrame([row_before, row_after]).set_index("split")
print("\nLogistic Regression — same features, same metric, two splits:\n")
print(comparison.round(3).to_string())
print(
    f"\nGap (BEFORE − AFTER) Precision@50: "
    f"{row_before['precision_at_50'] - row_after['precision_at_50']:+.3f}"
)
print(
    f"GroupKFold mean base rate: {row_after['base_rate']:.3f} "
    f"(mean test clients / fold ≈ {row_after['n_test_clients']:.1f})"
)
print("\nPer-fold AFTER detail:")
print(
    fold_df[
        ["fold", "n_test", "n_test_clients", "base_rate", "precision_at_10", "precision_at_20", "precision_at_50"]
    ]
    .round(3)
    .to_string(index=False)
)

OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)


Frame n = 52,001 | clients = 41
Slice: features=2026-03+2026-04; label=2026-05 | base rate = 0.515

Logistic Regression — same features, same metric, two splits:

                                 n_train  n_test  n_test_clients  base_rate  precision_at_10  precision_at_20  precision_at_50
split                                                                                                                         
BEFORE: random page split          39000   13001           38.00      0.515              1.0              1.0            0.980
AFTER: client GroupKFold (mean)    39000   13000           10.25      0.513              1.0              1.0            0.945

Gap (BEFORE − AFTER) Precision@50: +0.035
GroupKFold mean base rate: 0.513 (mean test clients / fold ≈ 10.2)

Per-fold AFTER detail:
 fold  n_test  n_test_clients  base_rate  precision_at_10  precision_at_20  precision_at_50
    1   13566               1      0.630              1.0              1.0             0.98
    2   128

## 3. Leakage audit

**Final feature set:** `imp_prev`, `ctr_prev`, `pos_avg_prev`, `engagement_rate_prev`, `position_tier`

**Attack checklist**

| Risk | Status |
|---|---|
| Label-derived / future features (`ctr_next`, `ctr_gap_next`, May tier median) | Excluded — trap below proves why |
| Past `ctr_gap` / `tier_median_ctr` | Baseline-only (ML-07); not a model feature (keeps the comparison fair) |
| Product flags / health scores | Not in warehouse — not used |
| IDs as features | `client_hash_id` / `content_hash_id` for split/join only |
| Future / overlapping windows | Features = Mar–Apr only; label = May; no June / `_sample`; no `fact_content_query_90d` |
| GA4 zeros | Engagement only when `ga4_data_available IS TRUE`; else filled 0 after filter |

**Trap test** shallow DecisionTree on past numeric features, once honest, once with May `ctr_gap_next` (label-window sibling). If AUC jumps toward 1.0, the harness works and outcome-window columns are illegal as features.

**Failures:** top-ranked OOS pages (client GroupKFold) that are not May underperformers (false alarms).


In [4]:
# --- Leakage trap: shallow tree on past feats ± May ctr_gap_next ---
honest_num = ["imp_prev", "ctr_prev", "pos_avg_prev", "engagement_rate_prev"]
X_honest = model_df[honest_num]
y_all = model_df[LABEL]
X_tr, X_te, y_tr, y_te = train_test_split(
    X_honest, y_all, test_size=0.25, random_state=RANDOM_STATE, stratify=y_all
)

tree_honest = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
tree_honest.fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, tree_honest.predict_proba(X_te)[:, 1])

X_leaky = X_honest.assign(ctr_gap_next=model_df["ctr_gap_next"])
tree_leaky = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE)
tree_leaky.fit(X_leaky.loc[X_tr.index], y_tr)
auc_leaky = roc_auc_score(y_te, tree_leaky.predict_proba(X_leaky.loc[X_te.index])[:, 1])

print("LEAKAGE TRAP (DecisionTree max_depth=2)")
print(f"Honest ROC AUC (past features only):     {auc_honest:.3f}")
print(f"Leaky ROC AUC (with May ctr_gap_next):   {auc_leaky:.3f}")
print(
    "Verdict: "
    + (
        "May ctr_gap_next leaks the label — keep ALL outcome-window columns OUT of features. "
        "Past ctr_gap stays baseline-only (not a model feature)."
        if auc_leaky - auc_honest > 0.02
        else "still exclude outcome-window columns; harness should jump if leaky feat is real."
    )
)
print()

# --- Failure examples from OOS client GroupKFold scores ---
fail = oos_eligible[oos_eligible[LABEL] == 0].nlargest(5, "model_score")[
    [
        "content_hash_id",
        "imp_prev",
        "ctr_prev",
        "position_tier",
        "baseline_score",
        "model_score",
        "ctr_next",
        LABEL,
    ]
]
print("Top-5 FALSE ALARMS on client GroupKFold OOS (high model score, May label=0):")
print(fail.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print(
    "Reading: pages the model ranks as urgent from past features, "
    "but in May they are NOT below-tier-median underperformers (label=0). "
    "A reviewer opening them first would waste time — classic false alarm cost."
)

trap = {
    "auc_honest": float(auc_honest),
    "auc_leaky": float(auc_leaky),
    "leaky_feature": "ctr_gap_next",
    "n_false_alarm_examples": int(len(fail)),
}
trap_path = OUT / "validation_leakage_trap.json"
trap_path.write_text(json.dumps(trap, indent=2))
print(f"\nSaved → {trap_path}")


LEAKAGE TRAP (DecisionTree max_depth=2)
Honest ROC AUC (past features only):     0.759
Leaky ROC AUC (with May ctr_gap_next):   1.000
Verdict: May ctr_gap_next leaks the label — keep ALL outcome-window columns OUT of features. Past ctr_gap stays baseline-only (not a model feature).

Top-5 FALSE ALARMS on client GroupKFold OOS (high model score, May label=0):
         content_hash_id   imp_prev  ctr_prev position_tier  baseline_score  model_score  ctr_next  is_ctr_underperformer
content_a8998878909cee77 28941.0000    0.0138        page_1          0.2023       0.9112    0.5373                      0
content_83167156f76e33e5 16340.0000    0.0184        page_1          0.1977       0.9015    0.4651                      0
content_2ba481a694613001  1451.0000    0.0000        page_1          0.2161       0.8997    0.3306                      0
content_87b9188513d77529 26046.0000    0.0269        page_1          0.1892       0.8992    0.3697                      0
content_6ac93124c81a6993 2028

## 4. Claim rewrite

**Bold (overreaches):**
> Our model beats the baseline and finds the pages that need a CTR fix.

Problems: "beats" without naming the split/metric; "need a CTR fix" sounds causal — we never tested that fixing the page raises clicks.

**Safe rewrite:**
> On Mar–Apr → May pages (`imp_prev >= 500`), under **client GroupKFold**, logistic regression **measured** mean Precision@K above the base rate and above the past-`ctr_gap` baseline (see numbers in the cell below). A random page split can look different — we report the **grouped** number when claiming skill. Week-5 **observed** the model surfaces high-stake underperformers that the fixed ML-07 `ctr_gap` rule ranks much lower. This is **decision-support** ranking, not proof that editing a page will raise clicks.

**Also safe to say:** always print base rate next to Precision@K; never claim Google outcomes or causal CTR lifts.


In [5]:
# Receipts for the claim rewrite — honest split numbers only
claim = {
    "bold_claim": "Our model beats the baseline and finds the pages that need a CTR fix.",
    "safe_claim": (
        "On Mar–Apr → May pages (imp_prev >= 500), under client GroupKFold, logistic "
        "regression measured mean Precision@K above the base rate and above the "
        "past-ctr_gap baseline (see numbers in the cell below). A random page split "
        "can look different — we report the grouped number when claiming skill. "
        "Week-5 observed the model surfaces high-stake underperformers that the fixed "
        "ML-07 ctr_gap rule ranks much lower. This is decision-support ranking, not "
        "proof that editing a page will raise clicks."
    ),
    "honest_split": row_after,
    "random_split_for_gap_only": row_before,
}
print("BOLD:")
print(" ", claim["bold_claim"])
print()
print("SAFE:")
print(" ", claim["safe_claim"])
print()
print("Honest metrics (AFTER: client GroupKFold mean):")
for k, v in row_after.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

print("\nRandom-split metrics (BEFORE — for the gap only):")
for k, v in row_before.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")


def _split_row(row):
    return {
        k: (
            float(v)
            if isinstance(v, (float, np.floating))
            else int(v)
            if isinstance(v, (int, np.integer))
            else v
        )
        for k, v in row.items()
    }


payload = {
    "bold_claim": claim["bold_claim"],
    "safe_claim": claim["safe_claim"],
    "slice": "features=2026-03+2026-04; label=2026-05",
    "population": f"imp_prev>={IMP_FLOOR}",
    "honest_split": _split_row(row_after),
    "random_split_for_gap_only": _split_row(row_before),
}
claim_path = OUT / "validation_claim_rewrite.json"
claim_path.write_text(json.dumps(payload, indent=2))
print(f"\nSaved → {claim_path}")


BOLD:
  Our model beats the baseline and finds the pages that need a CTR fix.

SAFE:
  On Mar–Apr → May pages (imp_prev >= 500), under client GroupKFold, logistic regression measured mean Precision@K above the base rate and above the past-ctr_gap baseline (see numbers in the cell below). A random page split can look different — we report the grouped number when claiming skill. Week-5 observed the model surfaces high-stake underperformers that the fixed ML-07 ctr_gap rule ranks much lower. This is decision-support ranking, not proof that editing a page will raise clicks.

Honest metrics (AFTER: client GroupKFold mean):
  split: AFTER: client GroupKFold (mean)
  n_train: 39000
  n_test: 13000
  n_test_clients: 10.250
  base_rate: 0.513
  precision_at_10: 1.000
  precision_at_20: 1.000
  precision_at_50: 0.945

Random-split metrics (BEFORE — for the gap only):
  split: BEFORE: random page split
  n_train: 39000
  n_test: 13001
  n_test_clients: 38
  base_rate: 0.515
  precision_at_10: 1.0